# Symbolic calculation

This notebook contains the exercises of the symbolic calculation lecture part 5.

In [1]:
import sympy as sym
import math
from typing import List, Dict
import graphviz
import numbers
from abc import ABC,abstractmethod

x,y,z,t,n,c,w = sym.symbols("x y z t n c w")

# AST complex processing

## Exercise 5.1

In this exercise, we will discover how SymPy expressions are represented as AST and how they could be rendered.

* Declare the following expressions with and without by preventing their evaluation/simplification at creation when needed and possible:
  * $x + 2y$
  * $\sin( w * x + b)$
  * $\frac{ x+1} {y+3}$
  * $\sum \limits _{n=1} ^{\infty}\frac{1} {n^4}$
  * $\sum \limits _{n=1} ^{\infty}\frac{1} {\sin^2(n)}$
  * $\frac{d}{dt}(t^2+1)$
  * $\frac{\partial^2}{\partial y \partial x} (xy + e^{xy}) $
  * $\int _{0} ^{1} (x+1)^2 dx$
* For each of them, observe their trees with the srepr, dotprint/graphviz.Source/view, latex functions

In [2]:
b = sym.symbols("b")

def show(e):
    """Arbre d'une expression : srepr, source graphviz (dot) et LaTeX."""
    print("srepr :", sym.srepr(e))
    print("latex :", sym.latex(e))
    print("dot   :", sym.dotprint(e).splitlines()[0:4], "... (%d lignes)" % len(sym.dotprint(e).splitlines()))
    return graphviz.Source(sym.dotprint(e))   # rendu SVG si l'exécutable 'dot' est installé

# x + 2y : avec / sans évaluation (ici identique car rien à simplifier)
e1 = x + 2*y
e1_ne = sym.Add(x, sym.Mul(2, y, evaluate=False), evaluate=False)
show(e1); show(e1_ne);

srepr : Add(Symbol('x'), Mul(Integer(2), Symbol('y')))
latex : x + 2 y
dot   : ['digraph{', '', '# Graph style', '"ordering"="out"'] ... (25 lignes)
srepr : Add(Symbol('x'), Mul(Integer(2), Symbol('y')))
latex : x + 2 y
dot   : ['digraph{', '', '# Graph style', '"ordering"="out"'] ... (25 lignes)


In [3]:
e2 = sym.sin(w*x + b)
e2_ne = sym.sin(sym.Add(sym.Mul(w, x, evaluate=False), b, evaluate=False), evaluate=False)
show(e2); show(e2_ne);

srepr : sin(Add(Symbol('b'), Mul(Symbol('w'), Symbol('x'))))
latex : \sin{\left(b + w x \right)}
dot   : ['digraph{', '', '# Graph style', '"ordering"="out"'] ... (27 lignes)
srepr : sin(Add(Symbol('b'), Mul(Symbol('w'), Symbol('x'))))
latex : \sin{\left(b + w x \right)}
dot   : ['digraph{', '', '# Graph style', '"ordering"="out"'] ... (27 lignes)


In [4]:
e3 = (x + 1)/(y + 3)
e3_ne = sym.Mul(x + 1, sym.Pow(y + 3, -1, evaluate=False), evaluate=False)
show(e3); show(e3_ne);

srepr : Mul(Add(Symbol('x'), Integer(1)), Pow(Add(Symbol('y'), Integer(3)), Integer(-1)))
latex : \frac{x + 1}{y + 3}
dot   : ['digraph{', '', '# Graph style', '"ordering"="out"'] ... (33 lignes)
srepr : Mul(Add(Symbol('x'), Integer(1)), Pow(Add(Symbol('y'), Integer(3)), Integer(-1)))
latex : \frac{x + 1}{y + 3}
dot   : ['digraph{', '', '# Graph style', '"ordering"="out"'] ... (33 lignes)


In [5]:
# Sum est paresseuse : l'expression n'est évaluée qu'avec doit()
e4 = sym.Sum(1/n**4, (n, 1, sym.oo))
e5 = sym.Sum(1/sym.sin(n)**2, (n, 1, sym.oo))
show(e4); print("evalué :", e4.doit())
show(e5); print("evalué :", e5.doit())

srepr : Sum(Pow(Symbol('n'), Integer(-4)), Tuple(Symbol('n'), Integer(1), oo))
latex : \sum_{n=1}^{\infty} \frac{1}{n^{4}}
dot   : ['digraph{', '', '# Graph style', '"ordering"="out"'] ... (31 lignes)
evalué : pi**4/90
srepr : Sum(Pow(sin(Symbol('n')), Integer(-2)), Tuple(Symbol('n'), Integer(1), oo))
latex : \sum_{n=1}^{\infty} \frac{1}{\sin^{2}{\left(n \right)}}
dot   : ['digraph{', '', '# Graph style', '"ordering"="out"'] ... (33 lignes)
evalué : Sum(sin(n)**(-2), (n, 1, oo))


In [6]:
e6 = sym.Derivative(t**2 + 1, t)            # non évalué
print("évalué :", e6.doit())
e7 = sym.Derivative(x*y + sym.exp(x*y), y, x)
print("évalué :", e7.doit())
show(e6); show(e7);

évalué : 2*t
évalué : x*y*exp(x*y) + exp(x*y) + 1
srepr : Derivative(Add(Pow(Symbol('t'), Integer(2)), Integer(1)), Tuple(Symbol('t'), Integer(1)))
latex : \frac{d}{d t} \left(t^{2} + 1\right)
dot   : ['digraph{', '', '# Graph style', '"ordering"="out"'] ... (33 lignes)
srepr : Derivative(Add(Mul(Symbol('x'), Symbol('y')), exp(Mul(Symbol('x'), Symbol('y')))), Tuple(Symbol('y'), Integer(1)), Tuple(Symbol('x'), Integer(1)))
latex : \frac{\partial^{2}}{\partial x\partial y} \left(x y + e^{x y}\right)
dot   : ['digraph{', '', '# Graph style', '"ordering"="out"'] ... (45 lignes)


In [7]:
e8 = sym.Integral((x + 1)**2, (x, 0, 1))
print("évalué :", e8.doit())
show(e8);

évalué : 7/3
srepr : Integral(Pow(Add(Symbol('x'), Integer(1)), Integer(2)), Tuple(Symbol('x'), Integer(0), Integer(1)))
latex : \int\limits_{0}^{1} \left(x + 1\right)^{2}\, dx
dot   : ['digraph{', '', '# Graph style', '"ordering"="out"'] ... (35 lignes)


In [8]:
# Comparaison arbre évalué / non évalué pour 1+1 : Add(1, 1) devient 2 immédiatement
print(sym.srepr(sym.Add(1, 1)), "|", sym.srepr(sym.Add(1, 1, evaluate=False)))
print(sym.srepr(sym.Mul(x, x)), "|", sym.srepr(sym.Mul(x, x, evaluate=False)))

Integer(2) | Add(Integer(1), Integer(1))
Pow(Symbol('x'), Integer(2)) | Mul(Symbol('x'), Symbol('x'))


## Exercise 5.2

We will experiment with standard polish notation for an AST. Consider that a mathematical expression is a tree (AST) and could be equivalently represented as a sequence with the operator first, followed by the operands:
* X+4 as [+,x, 4]
* \sin(xy) as [\sin,*,x,y]

* Declare a map that gives the corresponding string of a SymPy AST constructor
  * Mul() -> "*"
  * Add() -> "+"
  * ...
* Code a recursive function ast2polish that transforms a SymPy ast as a list of symbols satisfying the polish notation. In case of leaf values (numbers.Number, sym.Integer ...) and symbol, the function returns a singleton list with the input ast unchanged.
  * 1+2 -> ["+", 1,2]
 
Hints:
  * The extend method merges two lists
  * The function isinstance check that an instance is a subtype of a type
  * The instance attribute __class__ returns the type of an instance


In [9]:
polish_map = {sym.Mul: "*", sym.Add: "+", sym.Pow: "**",
              sym.sin: "sin", sym.cos: "cos", sym.tan: "tan",
              sym.exp: "exp", sym.log: "log"}

def ast2polish(ast):
    if isinstance(ast, (numbers.Number, sym.Symbol)):
        return [ast]
    op = polish_map.get(ast.func, ast.func.__name__)
    res = [op]
    for a in ast.args:
        res.extend(ast2polish(a))
    return res

* Test it on the following expressions:
  * X + y
  * 4 + yx
  * $\sin( xy + \frac{2} {5} )y$
  * $x^2 + xyz + 5$. Oberve that + and * are Nary operators.

In [10]:
ast2polish(x + y)

['+', x, y]

In [11]:
ast2polish(4 + y*x)

['+', 4, '*', x, y]

In [12]:
ast2polish(sym.sin(x*y + sym.Rational(2, 5))*y)

['*', y, 'sin', '+', 2/5, '*', x, y]

In [13]:
ast2polish(x**2 + x*y*z + 5)   # + et * sont n-aires : tous les opérandes à la suite

['+', 5, '**', x, 2, '*', x, y, z]

* Transforms x + 2zt and x + 2z + t by preventing the reordering or simplification of the expression (Hint: Add or Mul with parameter evaluate=False)
* Is is satisfying?
* How to solve this problem in case we would like to code the reverse function polish2ast?
* Code a function ast2polish2 that solves the problem

In [14]:
e_a = sym.Add(x, sym.Mul(2, z, t, evaluate=False), evaluate=False)          # x + 2zt
e_b = sym.Add(x, sym.Mul(2, z, evaluate=False), t, evaluate=False)          # x + 2z + t
print(ast2polish(e_a))
print(ast2polish(e_b))

['+', x, '*', 2, z, t]
['+', x, '*', 2, z, t]


In [15]:
# Les deux listes sont identiques alors que les arbres diffèrent : sans l'arité des opérateurs,
# la notation polonaise d'un opérateur n-aire est ambiguë.
print(ast2polish(e_a) == ast2polish(e_b), e_a == e_b)

True False


In [16]:
def ast2polish2(ast):
    """Comme ast2polish, mais l'opérateur est suivi de son arité : (op, nb_args)."""
    if isinstance(ast, (numbers.Number, sym.Symbol)):
        return [ast]
    res = [(polish_map.get(ast.func, ast.func.__name__), len(ast.args))]
    for a in ast.args:
        res.extend(ast2polish2(a))
    return res

print(ast2polish2(e_a))
print(ast2polish2(e_b))
print(ast2polish2(e_a) == ast2polish2(e_b))

# Réciproque (optionnel) : l'arité permet de savoir combien d'opérandes consommer
inv_map = {v: k for k, v in polish_map.items()}

def polish2ast2(lst):
    def parse(i):
        tok = lst[i]
        if isinstance(tok, tuple):
            op, arity = tok
            i += 1
            args = []
            for _ in range(arity):
                a, i = parse(i)
                args.append(a)
            return inv_map[op](*args, evaluate=False) if op in inv_map and inv_map[op] in (sym.Add, sym.Mul, sym.Pow) else inv_map[op](*args), i
        return tok, i + 1
    return parse(0)[0]

print(polish2ast2(ast2polish2(e_a)) == e_a, polish2ast2(ast2polish2(e_b)) == e_b)

[('+', 2), x, ('*', 3), 2, z, t]
[('+', 3), x, ('*', 2), 2, z, t]
False
True True


* Optional: think about a way to code the reverse function polish2ast2 of the previous function ast2polish2

## Exercise 5.3

## AST Traversal and processing

* Look at the following definition of the __AstProcessor__ class and its __AstDummyProcessor__ class implementation.

In [17]:
class AstProcessor:
    
    def process_node (self, node, parameter = None):
        methodRef = self.process_default
        if isinstance(node,numbers.Number):
            methodRef = self.process_Number
        else:
            try:
                methodName = "process_" + node.func.__name__
                methodRef = getattr(self, methodName)
            except AttributeError:
                methodRef = self.process_default
        return methodRef(node, parameter)

    def process_args (self, node, parameter=None) -> List[object]:
        return [ pa for a in node.args 
                for pa in [self.process_node(a, parameter)]]

    def process_default (self, node, parameter):
        print("process_default called", sym.srepr(node))
        return node

    def process_Number (self, node, parameter):
        print("process_Number called", node)
        return node


class AstDummyProcessor (AstProcessor):

    def process_Add(self, node, parameter:bool ):
        return sym.Mul( *node.args)

    def process_default (self, node, parameter:bool ):
        return node


We will code a subclass of AstProcessor named Evaluator that, given a valuation, i.e, a map that returns a value for every variable symbol occurring in the expression, evaluate the AST and returns its value. For example,
  * Evaluator.process_node( x+y , {x:1,y:2} ) returns 5.
In this exercise, we will not use the SymPy substitution subs function but try to code it in order to understand how it works.

* Code the needed methods (process_add,...) for evaluating $x-2*y$
* Test the evaluator with the {x:1, y:2} valuation.

In [18]:
class Evaluator(AstProcessor):
    """Évalue un AST, sans subs, à partir d'une valuation {symbole: valeur}."""

    def process_Number(self, node, parameter):
        return node

    def process_Symbol(self, node, parameter):
        return parameter[node]

    def process_Add(self, node, parameter):
        return sum(self.process_args(node, parameter))

    def process_Mul(self, node, parameter):
        return math.prod(self.process_args(node, parameter))

In [19]:
ev = Evaluator()
expr = x - 2*y            # = Add(x, Mul(-2, y))
print(sym.srepr(expr))
print(ev.process_node(expr, {x: 1, y: 2}), expr.subs({x: 1, y: 2}))

Add(Symbol('x'), Mul(Integer(-1), Integer(2), Symbol('y')))
-3 -3


* Code the required methods in ExtendedEvaluator for evaluating the following expression:
  * $ \frac{x} {y} * \sin ( x^3 ) - 4$

In [20]:
class ExtendedEvaluator(Evaluator):
    def process_Number(self, node, parameter):
        return float(node)

    def process_Pow(self, node, parameter):
        base, exp = self.process_args(node, parameter)
        return base**exp

    def process_sin(self, node, parameter):
        return math.sin(self.process_args(node, parameter)[0])

In [21]:
ee = ExtendedEvaluator()
expr2 = x/y*sym.sin(x**3) - 4
print(sym.srepr(expr2))
val = ee.process_node(expr2, {x: 1, y: 2})
print(val, float(expr2.subs({x: 1, y: 2})))

Add(Mul(Symbol('x'), Pow(Symbol('y'), Integer(-1)), sin(Pow(Symbol('x'), Integer(3)))), Integer(-4))
-3.5792645075960516 -3.5792645075960516


## Exercise 5.4

## Derivation and processing

In this exercise, we will code a processor that calculates the derivative of an expression without using the derivative set of derivation functions in SymPy. The differentiation variable will be given when the derivator processor is instantiated.

* Code a subclass of __ASTProcessor__ (see Exercise 5.2) named __Derivator__ with a constructor setting the derivative variable
* Code a derive and derive_args methods that take an expression and returns the derivative expression according to the derivative variable
* Code the required methods in the Derivator class for calculating the basic derivation according to x of the following expressions:
  * $10$ (process_Number methods)
  * $x$ (process_Symbol method)
  * $y$
  * $x + 6$ (process_Add function)


In [22]:
class Derivator(AstProcessor):
    """Dérive un AST par rapport à une variable, sans utiliser sym.diff."""

    def __init__(self, variable):
        self.variable = variable

    def derive(self, expr):
        return self.process_node(expr)

    def derive_args(self, node):
        return self.process_args(node)

    def process_Number(self, node, parameter=None):
        return sym.Integer(0)

    def process_Symbol(self, node, parameter=None):
        return sym.Integer(1) if node == self.variable else sym.Integer(0)

    def process_Add(self, node, parameter=None):
        return sym.Add(*self.derive_args(node))

In [23]:
d = Derivator(x)
d.derive(sym.Integer(10))

0

In [24]:
d.derive(x)

1

In [25]:
d.derive(y)

0

In [26]:
d.derive(x + 6)

1

The SymPy AST node is composed of 2 attributes, __func__ (constructor of node) and __args__ (list of sub-nodes). Recall the following properties:
* Node.func( *node.args ) = node
* $\frac{df(x)g(x)} {dx}  = \frac{d f(x)} {dx} g(x) + f(x) \frac{d g(x)} {dx}$
* The duplication of a Python list or tuple is performed by list(myListOrTuple)

* Code the required method of the __Derivator__ class for derivating $3x$ expression.

Hints:
* Collect the derived arguments calculated using process_args method
* Iterate on every argument of the node:
  * Copy arguments in a mulArgs list
  * Change the ith element of mulArgs with the derivative of the corresponding argument of the node
  * Create a multiplicative term and append it in a addArgs list
* Use the addArgs to create a returned  addition term

In [27]:
class Derivator(Derivator):          # on enrichit la classe : règle du produit
    def process_Mul(self, node, parameter=None):
        dargs = self.derive_args(node)
        addArgs = []
        for i in range(len(node.args)):
            mulArgs = list(node.args)
            mulArgs[i] = dargs[i]
            addArgs.append(sym.Mul(*mulArgs))
        return sym.Add(*addArgs)

d = Derivator(x)
print(d.derive(3*x))

3


Code the required methods of the Derivator class for derivating the following expressions.
  * $2x + x^2$
  * $x \sin ( 3 x )$
  * $\frac{x + y} {x^2 - 1}$
  * $(1+\sin(x) )^2$

In [28]:
class Derivator(Derivator):          # puissance (exposant quelconque) et fonctions usuelles
    def process_Pow(self, node, parameter=None):
        u, v = node.args
        du, dv = self.derive_args(node)
        # d(u^v) = u^v (v' ln u + v u'/u)  ; se réduit à v u^(v-1) u' si v est constant
        if dv == 0:
            return v*u**(v - 1)*du
        return node*(dv*sym.log(u) + v*du/u)

    def process_sin(self, node, parameter=None):
        return sym.cos(node.args[0])*self.derive_args(node)[0]

    def process_cos(self, node, parameter=None):
        return -sym.sin(node.args[0])*self.derive_args(node)[0]

    def process_exp(self, node, parameter=None):
        return node*self.derive_args(node)[0]

    def process_log(self, node, parameter=None):
        return self.derive_args(node)[0]/node.args[0]

d = Derivator(x)
e = 2*x + x**2
print(d.derive(e), "|", sym.diff(e, x))

2*x + 2 | 2*x + 2


In [29]:
e = x*sym.sin(3*x)
print(d.derive(e), "|", sym.diff(e, x))

3*x*cos(3*x) + sin(3*x) | 3*x*cos(3*x) + sin(3*x)


In [30]:
e = (x + y)/(x**2 - 1)
r = d.derive(e)
print(r)
print(sym.simplify(r - sym.diff(e, x)))

-2*x*(x + y)/(x**2 - 1)**2 + 1/(x**2 - 1)
0


In [31]:
e = (1 + sym.sin(x))**2
r = d.derive(e)
print(r)
print(sym.simplify(r - sym.diff(e, x)))

(2*sin(x) + 2)*cos(x)


0


## Exercise 5.5

In this exercise, we will introduce heuristics for antiderivative calculation as a processor.

* Look at the following classes definition:
  * INtegrator is an abstract class
  * CompositeIntegrator is based on a set of Integrator. Look at its find_antiderivative method
  * PrimitiveSearcherBase is the base class for every heuristic antiderivative searcher that will be implemented hereafter.


In [32]:
class Integrator(ABC):

    @abstractmethod
    def findPrimitive( self, ast: object) -> object:
        pass
    

class CompositeIntegrator(Integrator):

    def __init__(self, variable: sym.Symbol):
        self.subIntegrators = set()
        self.variable = variable

    def declareSubIntegrator(self, integrator:Integrator):
        self.subIntegrators.add(integrator)
    
    def findPrimitive( self, ast: object) -> object:
        for s in self.subIntegrators:
            primitive = s.findPrimitive(ast)
            if primitive is not None:
                return primitive
        return None

class IntegratorBase(Integrator, AstProcessor):
    
    def __init__(self, compositeIntegrator: CompositeIntegrator):
        self.compositeIntegrator = compositeIntegrator
        self.compositeIntegrator.declareSubIntegrator(self)
        self.variable = compositeIntegrator.variable

    @abstractmethod
    def isConstant( self, ast: object) -> bool:
        pass
    
    def findPrimitive(self, ast: object) -> object:
        return self.process_node(ast)

    def process_default (self, node, parameter):
        return None
        


* Implement a __FundamentalIntegrator__ class inheriting from IntegratorBase that returns the antiderivative of fundamental functions: p, \sin(x), \cos(x), x, x**p  (p constant)
* The constuctor of __FundamentalIntegrator__ has a compositeIntegrator input parameter of type __CompositeIntegrator__. It calls the upper __IntegratorBase__ constructor in its body(hint: IntegratorBase.\__init__(self,compositeIntegrator))
* What is a constant expression when a derivation is calculated against a variable?
* Overload the isConstant method (look at its declaration in the __IntegratorBase__) that determines if an ast is constant. 
* Build a compositeIntegrator instance based on the x variable (dummy integration variables)
* Build a fundamentalIntegrator instance
* Try the compositeIntegrator and its find_antiderivative method with:
  * X
  * \sin(x)
  * X**5



In [33]:
class FundamentalIntegrator(IntegratorBase):
    """Primitives des fonctions fondamentales : p, sin, cos, x, x**p."""

    def __init__(self, compositeIntegrator):
        IntegratorBase.__init__(self, compositeIntegrator)

    # constante par rapport à la variable : l'expression ne contient pas la variable
    def isConstant(self, ast):
        return self.variable not in ast.free_symbols if hasattr(ast, "free_symbols") else True

    def findPrimitive(self, ast):
        if self.isConstant(ast):
            return ast*self.variable
        return self.process_node(ast)

    def process_Symbol(self, node, parameter=None):
        return self.variable**2/2 if node == self.variable else None

    def process_sin(self, node, parameter=None):
        return -sym.cos(self.variable) if node.args[0] == self.variable else None

    def process_cos(self, node, parameter=None):
        return sym.sin(self.variable) if node.args[0] == self.variable else None

    def process_Pow(self, node, parameter=None):
        base, p = node.args
        if base == self.variable and self.isConstant(p):
            return sym.log(self.variable) if p == -1 else self.variable**(p + 1)/(p + 1)
        return None

In [34]:
compositeIntegrator = CompositeIntegrator(x)

In [35]:
fundamentalIntegrator = FundamentalIntegrator(compositeIntegrator)

In [36]:
for f_ in (x, sym.sin(x), x**5, sym.Integer(7)):
    P_ = compositeIntegrator.findPrimitive(f_)
    print(f_, "->", P_, "| vérif d/dx:", sym.simplify(sym.diff(P_, x) - f_) == 0)

x -> x**2/2 | vérif d/dx: True
sin(x) -> -cos(x) | vérif d/dx: True
x**5 -> 

x**6/6 | vérif d/dx: True
7 -> 7*x | vérif d/dx: True


The __LinearIntegrator__ will take care of the linear properties of the integration operator:
  * $ \int \lambda f(x)dx = \lambda \int f(x)dx $
  * $\int (f(x)+g(x))dx = \int f(x)dx + \int g(x)dx$

* Implement the __LinearIntegrator__ class by following the same pattern than the __FundamentalIntegrator__ class for the inheritance and the constructor
* Implement an instance method isConstant that determines if an ast is constant (look herebefore)
* If f(x) follows the pattern a * f(x) where a is a constant (method process_Mul), deduce the primitive by applying the previous linear properties (Hint: use isConstant method)
* If $f(x) = g_1(x) + ...+ g_n(x)$ (method process_Add), deduce the primitive by applying the previous linear properties





In [37]:
class LinearIntegrator(IntegratorBase):
    """Linéarité : ∫ λ f = λ ∫ f  et  ∫ (f+g) = ∫ f + ∫ g."""

    def __init__(self, compositeIntegrator):
        IntegratorBase.__init__(self, compositeIntegrator)

    def isConstant(self, ast):
        return self.variable not in ast.free_symbols if hasattr(ast, "free_symbols") else True

    def process_Number(self, node, parameter=None):
        return None            # laissé au FundamentalIntegrator (la base renverrait le noeud)

    def process_Mul(self, node, parameter=None):
        consts = [a for a in node.args if self.isConstant(a)]
        rest = [a for a in node.args if not self.isConstant(a)]
        if not consts or not rest:
            return None
        prim = self.compositeIntegrator.findPrimitive(sym.Mul(*rest))
        return None if prim is None else sym.Mul(*consts)*prim

    def process_Add(self, node, parameter=None):
        prims = [self.compositeIntegrator.findPrimitive(a) for a in node.args]
        return None if any(p is None for p in prims) else sym.Add(*prims)

* Build a compositeIntegrator instance based on the x variable (dummy integration variables)
* Build a FundamentaIntegrator instance
* Build a LinearIntegrator instance
* Try the compositeIntegrator and its find_antiderivative method with:
  * 3*x
  * \sin(x) + x
  * 3*\sin(x) + 2*x


In [38]:
compositeIntegrator = CompositeIntegrator(x)

In [39]:
fundamentalIntegrator = FundamentalIntegrator(compositeIntegrator)

In [40]:
linearIntegrator = LinearIntegrator(compositeIntegrator)

In [41]:
for f_ in (3*x, sym.sin(x) + x, 3*sym.sin(x) + 2*x):
    P_ = compositeIntegrator.findPrimitive(f_)
    print(f_, "->", P_, "| vérif d/dx:", sym.simplify(sym.diff(P_, x) - f_) == 0)

3*x -> 3*x**2/2 | vérif d/dx: True
x + sin(x) -> x**2/2 - cos(x) | vérif d/dx: True
2*x + 3*sin(x) -> x**2 - 3*cos(x) | vérif d/dx: True


* Think about coding an heuristic class about changing of dummy variable
* Think about coding code an heuristic class about integration by part
* Propose different expressions for testing those heuristics


## Changing of variable

## Integration by parts

* What could you envision other integration heuristics?

In [42]:
# --- Heuristique 1 : changement de variable  ∫ f(u(x)) u'(x) dx = ∫ f(v) dv
class SubstitutionIntegrator(IntegratorBase):
    def isConstant(self, ast):
        return self.variable not in ast.free_symbols if hasattr(ast, "free_symbols") else True

    def process_Number(self, node, parameter=None):
        return None

    def process_Mul(self, node, parameter=None):
        v = sym.Dummy("v")
        # candidats u : arguments des fonctions et bases de puissances
        cands = {a for sub in sym.preorder_traversal(node)
                 if isinstance(sub, (sym.Function, sym.Pow)) for a in sub.args[:1]}
        for u in cands:
            if u == self.variable or self.isConstant(u):
                continue
            ratio = sym.simplify(node/sym.diff(u, self.variable)).subs(u, v)
            if self.variable in ratio.free_symbols:
                continue
            sub_comp = CompositeIntegrator(v)
            FundamentalIntegrator(sub_comp); LinearIntegrator(sub_comp)
            prim = sub_comp.findPrimitive(ratio)
            if prim is not None:
                return prim.subs(v, u)
        return None

# --- Heuristique 2 : intégration par parties  ∫ u v' = u v - ∫ u' v   (u polynôme)
class ByPartsIntegrator(IntegratorBase):
    depth = 0
    def isConstant(self, ast):
        return self.variable not in ast.free_symbols if hasattr(ast, "free_symbols") else True

    def process_Number(self, node, parameter=None):
        return None

    def process_Mul(self, node, parameter=None):
        if ByPartsIntegrator.depth > 4 or len(node.args) != 2:
            return None
        for u, dv in (node.args, node.args[::-1]):
            if u.is_polynomial(self.variable) and not self.isConstant(u):
                ByPartsIntegrator.depth += 1
                try:
                    v_ = self.compositeIntegrator.findPrimitive(dv)       # ∫ dv
                    if v_ is None:
                        continue
                    rest = self.compositeIntegrator.findPrimitive(sym.expand(sym.diff(u, self.variable)*v_))
                    if rest is not None:
                        return sym.expand(u*v_ - rest)
                finally:
                    ByPartsIntegrator.depth -= 1
        return None

comp = CompositeIntegrator(x)
FundamentalIntegrator(comp); LinearIntegrator(comp); SubstitutionIntegrator(comp); ByPartsIntegrator(comp)
tests = [x*sym.cos(x**2), 2*x*sym.sin(x**2), x*sym.cos(x), x**2*sym.sin(x), sym.cos(3*x)*3]
for f_ in tests:
    P_ = comp.findPrimitive(f_)
    ok = P_ is not None and sym.simplify(sym.diff(P_, x) - f_) == 0
    print(f_, "->", P_, "| vérif d/dx:", ok)

x*cos(x**2) -> sin(x**2)/2 | vérif d/dx: True
2*x*sin(x**2) -> -cos(x**2) | vérif d/dx: True
x*cos(x) -> x*sin(x) + cos(x) | vérif d/dx: True
x**2*sin(x) -> -x**2*cos(x) + 2*x*sin(x) + 2*cos(x) | vérif d/dx: True
3*cos(3*x) -> sin(3*x) | vérif d/dx: True
